# **Filter-Based Methods:**

- Correlation-Based Analysis:
	- Pearson Correlation
	- ~~Kendall Correlation~~
	- ~~Spearman Correlation~~
- Mutual Information:
	- classification (for discrete target variables)
	- regression (for continuous target variables)
- ~~Chi-Squared Test~~
- Variance Threshold

In Feature Selection we're going to find a subset of the original features that make our ml task more efficient and accurate. Filter-based methods are a kind of approach in feature selection. They are the first baseline that we start using them when we wanna do a feature selection task.

**Pros:**
- Extendable for large datasets cause they are not computationaly expensive.
- Great baseline approach to reduce the dimensionality for more advanced methods that are computationaly expensive.

**Cons:**
- Don't consider interactions between features.
- They are more a univariate analysis.

## **Correlation-Based Analysis:**

One of the commong correlation-based filtering techniques is `Pearson Correlation`:

> **Note:** Pearson correlation in useful for dataset with numerical target variable.

> **Note:** To deal with datasets that have categorical target variable we can use Mutual Information (MI).

> **Note:** To deal with dataset with mixture of categorical and numerical features and a categorical target variable it's better to use Chi-Squared Test.

In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as ms

from sklearn import datasets
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression

In [3]:
diabetes_data = load_diabetes()
diabetes = pd.concat(
    [
        pd.DataFrame(diabetes_data['data'], columns=diabetes_data['feature_names']),
        pd.Series(diabetes_data['target'], name="target")
	],
    axis=1
)
diabetes

NameError: name 'load_diabetes' is not defined

In [ ]:
method = "pearson"

pearson_corr = abs(diabetes.drop("target", axis=1).corrwith(diabetes['target'], method=method)).sort_values(ascending=False)
pd.DataFrame(pearson_corr).rename({0: method + "_correlation"}, axis=1)

,pearson_correlation
bmi,0.586450
s5,0.565883
bp,0.441482
s4,0.430453
s3,0.394789
s6,0.382483
s1,0.212022
age,0.187889
s2,0.174054
sex,0.043062


In [ ]:
def corr_analysis(
        df: pd.DataFrame,
        target_name: str,
        method: str = "pearson",
        threshold: float = 0
) -> pd.DataFrame:
    """Create a data frame of the Correlation analysis

    This function creates a correlation analysis data frame based_on the
    method you choose for the analysis and using threshold argument you can
    filter the results based-on the correlation value. 

    :param df: The input dataframe
    :param target_name: The target variable name
    :param method: The correlation analysis method name, defaults to "pearson"
    :param threshold: Threshold to filter the results, defaults to 0
    :return: The correlation analysis data frame
    """
    try:
        corr = abs(df.drop(target_name, axis=1).corrwith(
            df[target_name], method=method)).sort_values(ascending=False)
        corr_df = pd.DataFrame(corr).rename({0: f"{method}_correlation"}, axis=1)
        
        return corr_df.loc[corr_df[f"{method}_correlation"] > threshold]

    except ValueError:
        print("ValueError: Method argument not exist.")
        print("Methods to choose: ['pearson', 'kendall', 'spearman']")
        return pd.DataFrame()

In [ ]:
corr_analysis_selected_featuers = corr_analysis(diabetes, target_name="target", method="pearson")
corr_analysis_selected_featuers

,pearson_correlation
bmi,0.586450
s5,0.565883
bp,0.441482
s4,0.430453
s3,0.394789
s6,0.382483
s1,0.212022
age,0.187889
s2,0.174054
sex,0.043062


The problem with this method is thet pearson correlation analysis can only handle continuous target variable and it doesn't consider the feature interactions and is a univariate analysis.

## **Mutual Information:**

Mutual information is a statistic property that indicates a how much does a feature have impact on the target variable predictability. High mutual information value indicates that the amount of information obtained from the target variable by observing the feature variable is high.

In [ ]:
X, y = diabetes_data.data, diabetes_data.target
mi_scores = mutual_info_classif(X, y)

mi_df = pd.DataFrame(
    data=mi_scores, index=diabetes_data.feature_names, columns=["mi-score"]
).sort_values(by="mi-score", axis=0, ascending=False)
mi_df

,mi-score
sex,1.167597
s4,0.278124
s5,0.188932
bmi,0.155486
s3,0.151952
age,0.057340
bp,0.041064
s6,0.012799
s2,0.000000
s1,0.000000


In [ ]:
from sklearn.feature_selection import mutual_info_regression

X, y = diabetes_data.data, diabetes_data.target
mi_scores = mutual_info_regression(X, y)

mi_df = pd.DataFrame(
    data=mi_scores, index=diabetes_data.feature_names, columns=["mi-score"]
).sort_values(by="mi-score", axis=0, ascending=False)
mi_df

,mi-score
bmi,0.177793
s5,0.143528
s6,0.115270
s4,0.109336
s1,0.068373
bp,0.065519
s3,0.056102
sex,0.035511
s2,0.013265
age,0.000000


read scikit-learn documentation for: mutual_info_regression, mutual_info_classif

There are differences between mutual information calculation for discrete target variable and continuous target variable:

In [ ]:
mi_score_reg = mutual_info_regression(X, y, random_state=42)
pd.DataFrame(
    mi_score_reg,
    index=diabetes_data.feature_names,
    columns=["mi_score"]
).sort_values(by="mi_score", ascending=False)

,mi_score
bmi,0.175849
s5,0.145293
s6,0.110071
s4,0.107700
s3,0.066790
s1,0.065387
bp,0.055903
sex,0.017674
s2,0.012951
age,0.001469


In [ ]:
# data with discrete (categorical) features
wine_data = datasets.load_wine()
wine = pd.concat(
    [
        pd.DataFrame(wine_data.data, columns=wine_data.feature_names),
        pd.Series(wine_data.target, name="target")
	],
    axis=1
)
wine

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,13.71,5.65,2.45,20.5,95.0,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740.0,2
174,13.40,3.91,2.48,23.0,102.0,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750.0,2
175,13.27,4.28,2.26,20.0,120.0,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835.0,2
176,13.17,2.59,2.37,20.0,120.0,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840.0,2


In [ ]:
X, y = wine_data.data, wine_data.target

mi_score_clf = mutual_info_classif(X, y)
pd.concat(
    [
        pd.DataFrame(
			mi_score_clf, index=wine_data.feature_names, columns=['mi_score']
		).sort_values(by="mi_score", ascending=False),
        pd.Series(wine.drop("target", axis=1).var(), name="var")
	],
    axis=1
)

,mi_score,var
flavanoids,0.662467,0.997719
color_intensity,0.555141,5.374449
proline,0.549547,99166.717355
od280/od315_of_diluted_wines,0.515662,0.504086
alcohol,0.475769,0.659062
hue,0.466132,0.052245
total_phenols,0.421715,0.391690
proanthocyanins,0.292092,0.327595
malic_acid,0.282043,1.248015
alcalinity_of_ash,0.232577,11.152686


In [ ]:
titanic = sns.load_dataset("titanic")
titanic = titanic.dropna()
titanic

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
10,1,3,female,4.0,1,1,16.7000,S,Third,child,False,G,Southampton,yes,False
11,1,1,female,58.0,0,0,26.5500,S,First,woman,False,C,Southampton,yes,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
871,1,1,female,47.0,1,1,52.5542,S,First,woman,False,D,Southampton,yes,False
872,0,1,male,33.0,0,0,5.0000,S,First,man,True,B,Southampton,no,True
879,1,1,female,56.0,0,1,83.1583,C,First,woman,False,C,Cherbourg,yes,False
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True


In [ ]:
for feature in [feature for feature in titanic.columns if feature not in ["age", "fare"]]:
    titanic[feature] = titanic[feature].astype("category")

In [ ]:
titanic.info()

<class 'pandas.DataFrame'>
Index: 182 entries, 1 to 889
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     182 non-null    category
 1   pclass       182 non-null    category
 2   sex          182 non-null    category
 3   age          182 non-null    float64 
 4   sibsp        182 non-null    category
 5   parch        182 non-null    category
 6   fare         182 non-null    float64 
 7   embarked     182 non-null    category
 8   class        182 non-null    category
 9   who          182 non-null    category
 10  adult_male   182 non-null    category
 11  deck         182 non-null    category
 12  embark_town  182 non-null    category
 13  alive        182 non-null    category
 14  alone        182 non-null    category
dtypes: category(13), float64(2)
memory usage: 7.4 KB


In [ ]:
categorical_features = [feature for feature, dtype in zip(titanic.columns, titanic.dtypes) if dtype == "category"]
numerical_features = [feature for feature, dtype in zip(titanic.columns, titanic.dtypes) if dtype == "float64"]
for feature in categorical_features:
    titanic[feature] = titanic[feature].cat.codes

data_columns = [feature for feature in titanic.columns if feature not in ["survived", "age", "fare"]]
X_categorical = titanic[categorical_features]
X_numerical = titanic[numerical_features]
y = titanic['survived']

mi_score_cat = mutual_info_classif(X_categorical, y, random_state=42, discrete_features=True)
mi_score_num = mutual_info_classif(X_numerical, y, random_state=42, discrete_features=False)

In [ ]:
pd.concat(
    [
        pd.Series(mi_score_cat),
        pd.Series(mi_score_num)
	]
)

0     0.629977
1     0.006788
2     0.155846
3     0.007404
4     0.008055
5     0.005531
6     0.006788
7     0.191647
8     0.187167
9     0.013297
10    0.005531
11    0.629977
12    0.006223
0     0.071397
1     0.050340
dtype: float64

In [ ]:
from sklearn.feature_selection import f_classif

f_score, _=f_classif(wine_data.data, wine_data.target)
f_score

NameError: name 'wine_data' is not defined

## **Variance Threshold:**

In [4]:
from sklearn.feature_selection import VarianceThreshold

In [13]:
iris_data = datasets.load_iris()

X = iris_data.data
y = iris_data.target


var_threshold = VarianceThreshold().fit(X)

In [14]:
var_threshold.variances_

array([0.68112222, 0.18871289, 3.09550267, 0.57713289])

In [17]:
var_threshold.n_features_in_

4

In [25]:
var_df = pd.DataFrame(var_threshold.variances_, index=iris_data.feature_names, columns=["var"])
var_df.sort_values(by="var", ascending=False)

,var
petal length (cm),3.095503
sepal length (cm),0.681122
petal width (cm),0.577133
sepal width (cm),0.188713


We can analyze our data and based-on the data properties decide to remove features with lower variances than the threshold we choose manualy. Introduce bias to the model and needs domain expertise to evaluate our task and select the best variance threshold.